# Env: akami1_rnaseq(R)

In [1]:
# RNA-Seq 데이터 처리 관련 라이브러리
library(DESeq2)      # 차등 발현 분석
library(tximport)    # Salmon, Kallisto 등에서의 데이터 가져오기
library(RUVSeq)      # RNA-Seq 데이터 정규화
library(sva)         # 배치 효과 제거

# 생물학적 데이터 분석 관련 라이브러리
library(clusterProfiler) # 기능적 풍부도 분석
library(topGO)           # GO 분석
library(fgsea)           # GSEA 분석
library(ReactomePA)      # Reactome Pathway 분석
library(org.Mm.eg.db)    # 마우스 유전자 데이터베이스
library(org.Hs.eg.db)    # 인간 유전자 데이터베이스
library(biomaRt)         # Ensembl 데이터베이스 접근

# 시각화 관련 라이브러리
library(ggplot2)         # 데이터 시각화
library(ggrepel)         # ggplot에서 텍스트 레이블 정리
library(pheatmap)        # 히트맵 생성
library(enrichplot)      # 풍부도 분석 시각화
library(ggridges)        # 리지 플롯 생성
library(RColorBrewer)    # 색상 팔레트 제공

# 데이터 처리 및 분석 관련 라이브러리
library(dplyr)       # 데이터 조작 및 처리
library(tibble)      # 데이터 프레임 확장
library(readr)       # 빠른 데이터 읽기
library(readxl)      # 엑셀 파일 읽기
library(writexl)     # 엑셀 파일 쓰기 # write_excel
library(openxlsx)    # 엑셀 파일 쓰기 # wirte.excel (행 이름 포함 가능)

# 문자열 처리 관련 라이브러리
library(stringr)         # 문자열 처리

# # Ensembl 데이터베이스 접근을 위한 Mart 설정 (예시)
# gene_mart <- useMart(biomart = "ENSEMBL_MART_ENSEMBL", 
#                      dataset = "hsapiens_gene_ensembl", 
#                      host = "https://may2024.archive.ensembl.org")

Loading required package: S4Vectors

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min



Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Loading required package: GenomicRanges

Loading required package: GenomeInfoDb

Loa

# DEG based enrichment analysis

In [2]:
### DEG enrichment test

# SYMBOL → ENTREZID 변환
# KEGG와 Reactome 분석을 위해 SYMBOL을 ENTREZID로 변환하는 함수
convert_symbol_to_entrez <- function(symbols, orgdb,keytype) {
  conversion <- bitr(
    symbols,
    fromType = keytype,
    toType = "ENTREZID",
    OrgDb = orgdb
  )
  return(conversion)
}

# 빈 결과 생성
empty_enrichment_df <- function() {
  data.frame(
    ID = character(),
    Description = character(),
    GeneRatio = character(),
    BgRatio = character(),
    pvalue = numeric(),
    p.adjust = numeric(),
    qvalue = numeric(),
    geneID = character(),
    Count = integer(),
    stringsAsFactors = FALSE
  )
}

# description_enrichment_df() 함수는 enrichment 결과의 각 열에 대한 설명을 제공하는 데이터 프레임을 생성합니다.
description_enrichment_df <- function() {
  data.frame(
    Column = c("ID", "Description", "GeneRatio", "BgRatio", "pvalue", "p.adjust", "qvalue", "geneID", "Count"),
    Meaning = c(
        "Term ID",
        "Term name",
        "Ratio of DEGs annotated to this term (Intersected DEGs / QuerySize)",
        "Ratio of background genes annotated to this term (TermSize / Background gene set size)",
        "Raw enrichment p-value with Hypergeometric test",
        "Adjusted p-value (Benjamini-Hochberg method)",
        "Adjusted p-value (Estimated false discovery rate (FDR))",
        "List of DEGs in this term",
        "Number of matched DEGs with term"
    )
  )
}

# GO: BP, CC, MF 수행 (결과: list of 3 dataframes)
perform_go_enrichment <- function(genes, keytype, orgdb) {
  ontologies <- c("BP", "CC", "MF")
  go_results <- list()

  for (ont in ontologies) {
    ego <- enrichGO(
      gene          = genes,
      OrgDb         = orgdb,
      ont           = ont,
      keyType       = keytype,
      pAdjustMethod = "BH",
      readable      = TRUE
    )

    if (!is.null(ego) && nrow(as.data.frame(ego)) > 0) {
      result <- as.data.frame(ego)
      result_sorted <- result[order(result$p.adjust), ]  # p.adjust 기준 오름차순 정렬
    } else {
      result_sorted <- empty_enrichment_df()
    }
    go_results[[ont]] <- result_sorted
  }
  return(go_results)
}

# KEGG 분석 (SYMBOL 입력 지원)
# KEGG는 데이터베이스가 entrezID를 기본으로 사용하므로, SYMBOL을 ENTREZID로 변환 후 분석
perform_kegg_enrichment <- function(genes, keytype, orgdb, Org_code) {
  conv <- convert_symbol_to_entrez(genes, orgdb, keytype)
  if (nrow(conv) == 0) {
    return(empty_enrichment_df())
  }

  ekegg <- enrichKEGG(
    gene = unique(conv$ENTREZID),
    organism = Org_code,
    keyType = "ncbi-geneid",
    pAdjustMethod = "BH"
  )

  if (!is.null(ekegg) && nrow(as.data.frame(ekegg)) > 0) {
    ekegg_readable <- setReadable(ekegg, OrgDb = orgdb, keyType = "ENTREZID")
    result <- as.data.frame(ekegg_readable)
    result <- result[order(result$p.adjust), ]  # p.adjust 기준 정렬
  } else {
    result <- empty_enrichment_df()
  }
  return(result)
}

# Reactome 분석 (SYMBOL 입력 지원)
perform_reactome_enrichment <- function(genes, keytype, orgdb, organism) {
  conv <- convert_symbol_to_entrez(genes, orgdb, keytype)
  if (nrow(conv) == 0) {
    return(empty_enrichment_df())
  }

  erct <- enrichPathway(
    gene          = unique(conv$ENTREZID),
    organism      = organism, #human, mouse
    pAdjustMethod = "BH",
    readable      = TRUE
  )
  if (!is.null(erct) && nrow(as.data.frame(erct)) > 0) {
    result <- as.data.frame(erct)
    result_sorted <- result[order(result$p.adjust), ]  # p.adjust 기준 오름차순 정렬
  } else {
    result_sorted <- empty_enrichment_df()
  }

  return(result_sorted)
}

perform_custom_geneset_enrichment <- function(genes, keytype, Custom_geneset_gmt) {

  if (length(genes) == 0) return(empty_enrichment_df())

  # Custom Geneset gmt 불러오기기
  Custom_geneset <- read.gmt(Custom_geneset_gmt)

  # enrichment 수행
  enrich_res <- enricher(
    gene = genes,
    TERM2GENE = Custom_geneset,
    pAdjustMethod = "BH"
  )

  # 정리
  if (!is.null(enrich_res) && nrow(as.data.frame(enrich_res)) > 0) {
    enrich_df <- as.data.frame(enrich_res)
    return(enrich_df[order(enrich_df$p.adjust), ])
  } else {
    return(empty_enrichment_df())
  }
}


plot_DEG_enrichment_analysis <- function(df, Top_n, path, prefix, direction, genes, prefix_suffix = "GO_BP") {
  try({
    # 데이터 전처리
    df$Intersection <- as.numeric(str_split(df$GeneRatio, "/", simplify = TRUE)[, 1])
    df$QuerySize <- as.numeric(str_split(df$GeneRatio, "/", simplify = TRUE)[, 2])
    df$TermSize <- as.numeric(str_split(df$BgRatio, "/", simplify = TRUE)[, 1])
    df$generatio <- df$Intersection / df$QuerySize

    # 상위 GO term 선택
    max <- min(Top_n, nrow(df))
    topn_df <- df[order(df$p.adjust), ][1:max, ]

    # 플롯 크기 조정
    plot_size <- adjust_plot_size(topn_df)

    # PDF 저장
    pdf(file = paste0(path, "2.DEG_enrichment/1.figure/", prefix, "_", prefix_suffix, ".pdf"),
        width = plot_size$width + 5, height = plot_size$height)

    # ggplot 생성
    p <- ggplot(topn_df, aes(x = generatio, 
                              y = reorder(paste0(Description, " (size: ", TermSize, ") "), -log10(p.adjust)), 
                              size = Count, color = -log10(p.adjust))) +
      geom_point(alpha = 0.6) +
      scale_color_gradient(low = "blue", high = "red") +
      theme_minimal() +
      labs(title = paste0(prefix, "_", prefix_suffix), 
           x = "Intersection Size / Term Size", 
           y = "Term", 
           size = "Intersection Size", 
           color = "-Log p-value") +
      annotate("text", x = Inf, y = Inf, label = paste(direction, " regulated DEGs: ", length(genes)), 
               vjust = 2, hjust = 1, size = 3) +
      theme(axis.text = element_text(size = 16),
            plot.title = element_text(hjust = 0.5))

    # 플롯 출력
    print(p)
    dev.off()
  })
}


GO_DEG_analysis = function(path,data,Organism,Comparison_name,Keytype,Top_n = 40, Custom_geneset_gmt = NULL) {
    make_folder(path,"2.DEG_enrichment/")
    make_folder(path,"2.DEG_enrichment/0.excel/")
    make_folder(path,"2.DEG_enrichment/1.figure/")
    if (Organism == "human"){
        OrgDb = org.Hs.eg.db
        Org_code = "hsa"
        Hallmark = "/octopus/yeongjun.kim/Reference/Hallmark/h.all.v2025.1.Hs.symbols.gmt"
    } else if (Organism == "mouse") {
        OrgDb = org.Mm.eg.db
        Org_code = "mmu"
        Hallmark = "/octopus/yeongjun.kim/Reference/Hallmark/mh.all.v2025.1.Mm.symbols.gmt"
    }


    sets = c("UP", "DOWN")
    for (direction in sets) {
        all_results <- list()
        genes <- data$gene_id[data$diffexpressed == direction]
        prefix <- paste(Comparison_name, direction, sep = "__")

        # 각각 enrichment 수행
        go_res <- perform_go_enrichment(genes, Keytype, OrgDb)
        kegg_res <- perform_kegg_enrichment(genes, Keytype, OrgDb, Org_code)
        reactome_res <- perform_reactome_enrichment(genes, Keytype, OrgDb, Organism)
        hallmark_res <- perform_custom_geneset_enrichment(genes, Keytype, Hallmark)
        for (ont in names(go_res)) {
            GO_name = paste0("GO_", ont)
            all_results[[GO_name]] <- go_res[[ont]]
        }
        all_results[["KEGG"]] <- kegg_res
        all_results[["Reactome"]] <- reactome_res
        all_results[["Hallmark"]] <- hallmark_res
        if (!is.null(Custom_geneset_gmt)) {
            custom_res <- perform_custom_geneset_enrichment(genes, Keytype, Custom_geneset_gmt)
            all_results[["Custom"]] <- custom_res
        }
        all_results[["Column_Description"]] = description_enrichment_df()
        write_xlsx(all_results, paste0(path,"2.DEG_enrichment/0.excel/",prefix,".xlsx"))
        # 시각화 예시
        # GO_BP 시각화
        prefix_suffix = "GO_BP"
        plot_DEG_enrichment_analysis(all_results[[prefix_suffix]], Top_n, path, prefix, direction, genes,  prefix_suffix)
    }
}




# Gene Set Enrichment Analysis (GSEA)

In [ ]:
# SYMBOL → ENTREZID 변환
convert_vector_to_entrez <- function(genes, orgdb, keytype) {
  # 이름 추출
  original_ids <- names(genes)

  # 변환
  conversion <- bitr(
    original_ids,
    fromType = keytype,
    toType = "ENTREZID",
    OrgDb = orgdb
  )

  # 매칭된 ENTREZID만 추출
  matched <- conversion[conversion[[keytype]] %in% original_ids, ]
  
  # log2FC 값 매칭
  matched$log2FC <- genes[matched[[keytype]]]

  # ENTREZID를 이름으로 설정한 named vector 생성
  entrez_vector <- setNames(matched$log2FC, matched$ENTREZID)

  # 내림차순 정렬
  entrez_vector <- sort(entrez_vector, decreasing = TRUE)

  return(entrez_vector)
}

# 빈 결과 처리용 템플릿
empty_gsea_df <- function() {
  data.frame(
    ID = character(),
    Description = character(),
    setSize = character(),
    enrichmentScore = character(),
    NES = character(),
    pvalue = numeric(),
    p.adjust = numeric(),
    qvalue = numeric(),
    rank = character(),
    leading_edge = character(),
    core_enrichment = character(),
    stringsAsFactors = FALSE
  )
}

description_gsea_df <- function() {
  # GSEA 결과의 각 열에 대한 설명을 제공하는 데이터 프레임 생성
    data.frame(
    Column = c(
        "Reference",
        "ID",
        "Description",
        "setSize",
        "enrichmentScore",
        "NES",
        "pvalue",
        "p.adjust",
        "qvalue",
        "rank",
        "leading_edge",
        "core_enrichment"
    ),
    Meaning = c(
        "https://www.gsea-msigdb.org/gsea/doc/GSEAUserGuideFrame.html",
        "Term ID",
        "Term name",
        "Number of genes in the term",
        "Raw enrichment score (ES); reflects how concentrated gene ranks are at the top/bottom of the ranked list",
        "Normalized Enrichment Score (ES normalized by set size and permutations)",
        "The p-value of the ES is calculated using a permutation test",
        "Adjusted p-value (Benjamini-Hochberg method)",
        "Adjusted p-value (Estimated false discovery rate (FDR))",
        "Rank in the ordered gene list where the enrichment score peak occurs",
        paste0(
        "Summary of the leading edge subset contributing most to the enrichment score:\n",
        " - tags: % of gene hits before (for +ES) or after (for -ES) the peak\n",
        " - list: % of ranked genes before (for +ES) or after (for -ES) the peak\n",
        " - signal: combined enrichment strength"
        ),
        "The actual genes contributing to the enrichment signal (leading edge core genes)"
        )
    )
}
## plot 크기 조정 함수

# GSEA 결과 필터링 함수
filter_gsea_result <- function(df, pvalue_cutoff = 0.05, min_set_size = 10, max_set_size = 500) {
  result <- df %>%
    arrange(p.adjust) %>%
    filter(p.adjust < pvalue_cutoff, setSize >= min_set_size, setSize <= max_set_size)
  return(result)
}

adjust_plot_size <- function(top_df) {
  # 기본 크기
  width <- 8
  height <- 8
  
  # 레이블의 길이에 따라 너비를 조정
  max_label_length <- max(nchar(    as.character(top_df$Description)  ))
  counts = nrow(top_df)

  adjusted_width <- width + max_label_length * 0.1
  adjusted_height <- height - (20 - counts) * 0.3
  
  # 조정된 너비와 높이를 반환
  return(list(width = adjusted_width, height = adjusted_height))
}

select_top_GSEA_abs <- function(GSEA_result, top_n = 30) {
    ## 최대 top 30의 GO term에 대해 visualization 하는데,
    ## p.adjust로 나눠서 NES value로 정렬
    Len = nrow(GSEA_result)
    GSEA_result %>%
        data.frame() %>%
        arrange(p.adjust) %>%
        head(n = min(top_n,Len)) %>%
        arrange(NES) -> top_pathway_data
    top_pathway_data$description = paste0(top_pathway_data$Description," (setsize: ",top_pathway_data$setSize,") ")
    factor(top_pathway_data$description, levels = top_pathway_data$description) -> top_pathway_data$description
    return(top_pathway_data)
}

draw_gsea_summary_plot <- function(GSEA_result, prefix_suffix, path, Comparison_name, top_n = 30, pvalue_cutoff = 0.05, min_set_size = 10, max_set_size = 500) {
    # try({ # GSEA 결과가 없을 때를 대비한 예외 처리
        set.seed(116)
        sort_df <- GSEA_result %>%
                arrange(p.adjust)
        sort_df = filter_gsea_result(sort_df, pvalue_cutoff = pvalue_cutoff, min_set_size = min_set_size, max_set_size = max_set_size)
        top_df = select_top_GSEA_abs(sort_df, top_n = top_n)

        # pdf file format
        plot_size <- adjust_plot_size(top_df)
        pdf(file= paste0(path,"3.GSEA/1.figure/",Comparison_name,"_",prefix_suffix,".pdf"), width=plot_size$width, height=plot_size$height)
        
        #draw figure
        p = ggplot(top_df ) +
        geom_bar(aes(x = description, y = NES, fill = p.adjust),
                stat='identity') +
        scale_fill_continuous(low = "red", high = "blue") +
        coord_flip(ylim = c(-3.2, 3.2)) +
        scale_y_continuous(breaks = seq(-3,3,1)) +
        theme_bw(14) +
        labs(title=paste0(Comparison_name,"_",prefix_suffix),
                x = "GO Term") +  
        theme(plot.title = element_text(hjust = 0.5),
                axis.title.y = element_blank())

        print(p)
        dev.off()
    # })
}
### GSEA 함수

perform_go_gsea <- function(genes, keytype, orgdb, Comparison_name,path) {
  ontologies <- c("BP", "CC", "MF")
  go_results <- list()

  for (ont in ontologies) {
    ego <- gseGO(
      gene          = genes,
      OrgDb         = orgdb,
      ont           = ont,
      keyType       = keytype,
      pAdjustMethod = "BH"
    )

    if (!is.null(ego) && nrow(as.data.frame(ego)) > 0) {
      # gsea_go_readable <- setReadable(ego, OrgDb = org.Hs.eg.db, keyType = keytype)
      result <- as.data.frame(ego)
      result_sorted <- result[order(result$p.adjust), ]  # p.adjust 기준 오름차순 정렬
    } else {
      result_sorted <- empty_gsea_df()
    }
    go_results[[ont]] <- result_sorted
  }

  return(go_results)
}

# KEGG 분석 (SYMBOL 입력 지원)
perform_kegg_gsea <- function(genes, keytype, orgdb, Org_code) {
  conv <- convert_vector_to_entrez(genes, orgdb, keytype)

  ekegg <- gseKEGG(
    gene = conv,
    organism = Org_code,
    keyType = "ncbi-geneid",
    pAdjustMethod = "BH"
  )

  if (!is.null(ekegg) && nrow(as.data.frame(ekegg)) > 0) {
    ekegg_readable <- setReadable(ekegg, OrgDb = orgdb, keyType = "ENTREZID")
    result <- as.data.frame(ekegg_readable)
    result <- result[order(result$p.adjust), ]  # p.adjust 기준 정렬
  } else {
    result <- empty_gsea_df()
  }
  return(result)
}

# Reactome 분석 (결과: data.frame)
perform_reactome_gsea <- function(genes, keytype, orgdb, organism) {
  conv <- convert_vector_to_entrez(genes, orgdb, keytype)

  erct <- gsePathway(
    gene          = conv,
    organism      = organism, #human, mouse
    pAdjustMethod = "BH"
  )
  if (!is.null(erct) && nrow(as.data.frame(erct)) > 0) {
    ekegg_readable <- setReadable(erct, OrgDb = orgdb, keyType = "ENTREZID")
    result <- as.data.frame(ekegg_readable)
    result_sorted <- result[order(result$p.adjust), ]  # p.adjust 기준 오름차순 정렬
  } else {
    result_sorted <- empty_gsea_df()
  }

  return(result_sorted)
}

perform_custom_geneset_gsea <- function(genes, Custom_geneset_gmt) {

  if (length(genes) == 0) return(empty_gsea_df())

  # hallmark gene set 불러오기
  Custom_df <- read.gmt(Custom_geneset_gmt)

  # GSEA 실행
  gsea_res <- GSEA(
    geneList = genes,
    TERM2GENE = Custom_df,
    pAdjustMethod = "BH"
  )

  # 정리
  if (!is.null(gsea_res) && nrow(as.data.frame(gsea_res)) > 0) {
    gsea_df <- as.data.frame(gsea_res)
    return(gsea_df[order(gsea_df$p.adjust), ])
  } else {
    return(empty_gsea_df())
  }
}

run_specific_gsea_analysis <- function(path, data, Comparison_name, specific_geneset, pvalue_cutoff = 1) {
    try({
            if (is.null(specific_geneset)) {
        return(invisible(NULL))
        }

        Genes <- setNames(data$log2FoldChange, data$gene_id)
        Genes <- sort(Genes, decreasing = TRUE)

        hallmark_df <- read.gmt(specific_geneset)

        gsea_res <- GSEA(
            geneList = Genes,
            TERM2GENE = hallmark_df,
            pAdjustMethod = "BH",
            pvalueCutoff = pvalue_cutoff
        )

        draw_Specific_GSEA_plot(path = path, data = gsea_res, Name = Comparison_name)
    })

}

GSEA_analysis = function(path,data,Organism,Comparison_name,Keytype,Top_n = 20, Custom_geneset_gmt = NULL, prefix_suffix = "GO_BP") {
    make_folder(path,"3.GSEA/")
    make_folder(path,"3.GSEA/0.excel/")
    make_folder(path,"3.GSEA/1.figure/")
    #ready for data
    if (Organism == "human"){
        OrgDb = org.Hs.eg.db
        Org_code = "hsa"
        Hallmark = "/octopus/yeongjun.kim/Reference/Hallmark/h.all.v2025.1.Hs.symbols.gmt"
    } else if (Organism == "mouse") {
        OrgDb = org.Mm.eg.db
        Org_code = "mmu"
        Hallmark = "/octopus/yeongjun.kim/Reference/Hallmark/mh.all.v2025.1.Mm.symbols.gmt"
    }
    
    all_results = list()

    Genes <- setNames(data$log2FoldChange, data$gene_id)
    Genes <- sort(Genes, decreasing = TRUE)
    prefix <- Comparison_name ## 바꾸려면 바꾸면 된다

    # 각각 enrichment 수행
    go_res <- perform_go_gsea(Genes, Keytype, OrgDb)
    kegg_res <- perform_kegg_gsea(Genes, Keytype, OrgDb, Org_code)
    reactome_res <- perform_reactome_gsea(Genes, Keytype, OrgDb, Organism)
    hallmark_res <- perform_custom_geneset_gsea(Genes, Hallmark)


    for (ont in names(go_res)) {
        GO_name = paste0("GO_", ont)
        all_results[[GO_name]] <- go_res[[ont]]
    }
    all_results[["KEGG"]] <- kegg_res
    all_results[["Reactome"]] <- reactome_res
    all_results[["Hallmark"]] <- hallmark_res
    all_results[["Column_Description"]] = description_gsea_df()
    write_xlsx(all_results, paste0(path,"3.GSEA/0.excel/",prefix,".xlsx"))

    # 시각화 대표: default - GO_BP
    GSEA_result = all_results[[prefix_suffix]]
    draw_gsea_summary_plot(
        GSEA_result = GSEA_result,
        prefix_suffix = prefix_suffix,
        path = path,
        Comparison_name = Comparison_name,
        top_n = 30
    )
}
draw_Specific_GSEA_plot = function(path, data, Name, P.adjust = 0.05){
    Genesets <- data$ID
    for (Geneset in Genesets){
        try({
            # setting details
            Geneset_description = data$Description[data$ID == Geneset]
            NES = format(data$NES[data$ID == Geneset], digits = 4)
            FDR = format(data$p.adjust[data$ID == Geneset], scientific = TRUE, digits = 4)
            pvalue = format(data$pvalue[data$ID == Geneset], scientific = TRUE, digits = 4)
            setSize = data$setSize[data$ID == Geneset]
            
            # P.adjust threshold
            if (data$p.adjust[data$ID == Geneset] < P.adjust){
                color = "red"
            } else {
                color = "black"
            }
            
            # make folder and save plot
            make_folder(path, paste0("3.GSEA/"))
            make_folder(path, paste0("3.GSEA/2.GeneSet/"))
            make_folder(path, paste0("3.GSEA/2.GeneSet/", Geneset_description))
            pdf(file = paste0(path, "3.GSEA/2.GeneSet/", Geneset_description, "/", Name, ".pdf"), width = 5, height = 5)
            
            p = gseaplot2(data, geneSetID = Geneset, subplots = 1:3, title = "", ES = "line")
            
            p[[1]] = p[[1]] +
                annotate("text", x = Inf, y = Inf, label = paste0("p-value: ", pvalue), hjust = 1, vjust = 2, size = 3) + 
                annotate("text", x = Inf, y = Inf, label = paste0("FDR: ", FDR), hjust = 1, vjust = 4, size = 3, color = color) +
                annotate("text", x = Inf, y = Inf, label = paste0("NES: ", NES), hjust = 1, vjust = 6, size = 3) +
                labs(title = Name, subtitle = paste0(Geneset_description, "\n (setsize: ", setSize, ") "), y = "Running Enrichment Score") + 
                theme(plot.title = element_text(hjust = 0.5, size = 14),
                      plot.subtitle = element_text(hjust = 0.5, size = 12, color = color))
            
            print(p)
            dev.off()
        })
    }
}

# Visualization

In [9]:
plot_pca_with_condition_and_batch <- function(
  expr_matrix,
  col_data,
  result_path,
  file_name = "PCA_plot.png",
  plot_title = "PCA: TMM + Length normalized log expression",
  condition_col_1 = "sample",      # 색상으로 사용할 컬럼
  condition_col_2 = "Condition",   # 모양으로 사용할 컬럼
  gene_annotation = NULL,          # 매트릭스에 gene_name이 없을 때 fallback용
  point_alpha = 0.55,               # 포인트 투명도(0~1)
  sample_legend_ncol = 1,
  sample_legend_cex = 0.85,
  legend_width_ratio = 1.2,
  pc_num_1 = 1,
  pc_num_2 = 2,
  show_sample_labels = TRUE,
  sample_label_cex = 0.7,
  sample_label_pos = 3,
  sample_label_offset = 0.5
  ) {

  # 0) expr_matrix에서 annotation 컬럼(gene_id, gene_name) 분리
  annot_col_names <- intersect(c("gene_id", "gene_name"), colnames(expr_matrix))
  if (length(annot_col_names) > 0) {
    gene_annot_from_matrix <- expr_matrix[, annot_col_names, drop = FALSE]
    expr_numeric <- expr_matrix[, setdiff(colnames(expr_matrix), annot_col_names), drop = FALSE]
  } else {
    gene_annot_from_matrix <- NULL
    expr_numeric <- expr_matrix
  }

  # 1) 필수 컬럼 확인
  req_cols <- c(condition_col_1, condition_col_2, "sample")
  miss <- setdiff(req_cols, colnames(col_data))
  if (length(miss) > 0) {
    stop("col_data에 다음 컬럼이 필요합니다: ", paste(miss, collapse = ", "))
  }

  # 2) 샘플 일치성 및 정렬 (수치 컬럼만 기준)
  samples_expr <- colnames(expr_numeric)
  samples_meta <- as.character(col_data$sample)
  if (!all(samples_expr %in% samples_meta)) {
    stop("expr_matrix의 모든 샘플 컬럼명이 col_data$sample에 존재해야 합니다. ",
         "누락: ", paste(setdiff(samples_expr, samples_meta), collapse = ", "))
  }
  col_data <- col_data[match(samples_expr, samples_meta), , drop = FALSE]

  # 3) 색상: condition_col_1
  tol15 <- c(
    "#332288", "#88CCEE", "#44AA99", "#117733", "#999933",
    "#DDCC77", "#CC6677", "#882255", "#AA4499", "#817066",
    "#E69F00", "#D55E00", "#53377A", "#7F180D", "#00538A"
  )

  condition_1_vals <- as.character(col_data[[condition_col_1]])
  condition_1_levels <- unique(condition_1_vals)
  n_cond_1 <- length(condition_1_levels)
  palette_1 <- if (n_cond_1 <= length(tol15)) tol15[seq_len(n_cond_1)] else grDevices::colorRampPalette(tol15)(n_cond_1)
  condition_1_colors <- setNames(palette_1, condition_1_levels)
  condition_1_colors_alpha <- sapply(condition_1_colors, function(col) grDevices::adjustcolor(col, alpha.f = point_alpha))

  # 4) 모양: condition_col_2
  condition_2_vals <- as.character(col_data[[condition_col_2]])
  condition_2_levels <- unique(condition_2_vals)
  filled_pchs <- c(21, 22, 23, 24, 25, 0, 2, 5, 6, 13)
  condition_2_pch_vals <- rep(filled_pchs, length.out = length(condition_2_levels))
  condition_2_pch <- setNames(condition_2_pch_vals, condition_2_levels)

  # 5) PCA (수치 컬럼만 사용)
  pca_res <- prcomp(t(expr_numeric), center = TRUE, scale. = TRUE)
  imp <- summary(pca_res)$importance[2, c(pc_num_1, pc_num_2)]

  # 6) 저장 시작
  png(filename = file.path(result_path, file_name), width = 12, height = 8, units = "in", res = 300)
  on.exit(dev.off(), add = TRUE)

  # 7) 레이아웃: [플롯 | 범례]
  opar <- par(no.readonly = TRUE); on.exit(par(opar), add = TRUE)
  layout(matrix(c(1, 2), nrow = 1), widths = c(3, legend_width_ratio))

  ## 좌측: PCA 플롯
  par(mar = c(5, 5, 4, 1))
  point_pch <- condition_2_pch[condition_2_vals]
  point_bg  <- condition_1_colors_alpha[condition_1_vals]
  point_text_col <- condition_1_colors[condition_1_vals]

  plot(
    x = pca_res$x[, pc_num_1], y = pca_res$x[, pc_num_2],
    pch = point_pch,
    bg  = point_bg,
    col = "black", cex = 1.6,
    xlab = paste0("PC", pc_num_1, " (", round(100 * imp[1], 1), "%)"),
    ylab = paste0("PC", pc_num_2, " (", round(100 * imp[2], 1), "%)"),
    main = plot_title
  )

  # Sample 이름 텍스트
  if (show_sample_labels) {
    text(
      pca_res$x[, pc_num_1], pca_res$x[, pc_num_2],
      labels = as.character(col_data$sample),
      col = point_text_col,
      cex = sample_label_cex,
      pos = sample_label_pos,
      offset = sample_label_offset
    )
  }

  ## 우측: 범례 전용 패널
  par(mar = c(2, 1, 2, 1))
  plot.new()

  legend(
    "topleft",
    title  = paste0(condition_col_1, " (color)"),
    legend = condition_1_levels,
    pch    = 21,
    pt.bg  = condition_1_colors_alpha[condition_1_levels],
    col    = "black",
    pt.cex = 1.5,
    cex    = sample_legend_cex,
    bty    = "n",
    ncol   = sample_legend_ncol,
    x.intersp = 0.6, y.intersp = 0.9
  )

  legend(
    "bottomleft",
    title  = paste0(condition_col_2, " (shape)"),
    legend = condition_2_levels,
    pch    = condition_2_pch[condition_2_levels],
    pt.bg  = "grey85",
    col    = "black",
    pt.cex = 1.5,
    cex    = sample_legend_cex,
    bty    = "n",
    x.intersp = 0.8, y.intersp = 0.9
  )

  # 8) PCA loading matrix 저장
  loading_df <- as.data.frame(pca_res$rotation)

  # gene_id: 매트릭스에 포함된 경우 사용, 없으면 rownames에서 추출
  if (!is.null(gene_annot_from_matrix) && "gene_id" %in% colnames(gene_annot_from_matrix)) {
    loading_df$gene_id <- gene_annot_from_matrix$gene_id
    loading_df <- loading_df %>% relocate(gene_id, .before = 1)

    if ("gene_name" %in% colnames(gene_annot_from_matrix)) {
      loading_df$gene_name <- gene_annot_from_matrix$gene_name
      loading_df <- loading_df %>% relocate(gene_name, .after = gene_id)
    }
  } else {
    # fallback: rownames를 gene_id로 사용
    loading_df <- rownames_to_column(loading_df, var = "gene_id")
  }

  loading_file_name <- paste0(tools::file_path_sans_ext(file_name), "_loading.csv")
  write.csv(loading_df, file = file.path(result_path, loading_file_name), row.names = FALSE)
}

draw_volcanoplot = function(path,data,method,FC,pval,Comparison_name){
    make_folder(path,"1.volcano/")
    mycolors <- c("blue", "red", "black")
    names(mycolors) <- c("DOWN", "UP", "NO")
    Log10Pvalue = -log10(data[[method]])

    pdf(file= paste0(path,"1.volcano/",Comparison_name,".pdf"), width=7, height=5)

    plot = ggplot(data=data, aes(x=log2FoldChange, y=Log10Pvalue, col=diffexpressed)) + 
                geom_point() + 
                theme_minimal() +
                #geom_text_repel() +
                scale_color_manual(values=mycolors) +
                geom_vline(xintercept=c(-FC, FC), col="red") +
                geom_hline(yintercept=-log10(pval), col="red") + annotate("rect", xmin=Inf, xmax=Inf, ymin=Inf, ymax=Inf, fill="white", alpha=0.5) +
                annotate("text", x=Inf, y=Inf, label=paste("UP:", sum(data$diffexpressed == "UP")), vjust=3, hjust=1, size=3) + # sum이 원래 count 였음 24/11/30
                annotate("text", x=Inf, y=Inf, label=paste("DOWN:", sum(data$diffexpressed == "DOWN")), vjust=5, hjust=1, size=3) +
                ggtitle(Comparison_name) + 
                theme(plot.title = element_text(hjust = 0.5)) + 
                ylab("-Log10 adj P-value") + 
                xlab("Log2 Fold Change")
    print(plot)
    dev.off()
}

draw_volcanoplot_annot = function(path,data,method,FC,pval,Comparison_name){
    make_folder(path,"1.volcano_annot/")
    mycolors <- c("blue", "red", "black")
    names(mycolors) <- c("DOWN", "UP", "NO")
    Log10Pvalue = -log10(data[[method]])
    pdf(file= paste0(path,"1.volcano_annot/",Comparison_name,".pdf"), width=7, height=5)

    plot = ggplot(data=data, aes(x=log2FoldChange, y=Log10Pvalue, col=diffexpressed, label=delabel)) + ## , label=delabel 이거 붙으면 text 들어가는거야.
                geom_point() + 
                theme_minimal() +
                geom_text_repel() +
                scale_color_manual(values=mycolors) +
                geom_vline(xintercept=c(-FC, FC), col="red") +
                geom_hline(yintercept=-log10(pval), col="red") + annotate("rect", xmin=Inf, xmax=Inf, ymin=Inf, ymax=Inf, fill="white", alpha=0.5) +
                annotate("text", x=Inf, y=Inf, label=paste("UP:", sum(data$diffexpressed == "UP")), vjust=3, hjust=1, size=3) + # sum이 원래 count 였음 24/11/30
                annotate("text", x=Inf, y=Inf, label=paste("DOWN:", sum(data$diffexpressed == "DOWN")), vjust=5, hjust=1, size=3) +
                ggtitle(paste0(Comparison_name)) + 
                theme(plot.title = element_text(hjust = 0.5)) + 
                ylab("-Log10 adj P-value") + 
                xlab("Log2 Fold Change")
    print(plot)
    dev.off()
}

draw_volcanoplot_genes = function(path, data, method, FC, pval, Comparison_name, specific_genes = NULL){
    make_folder(path,"1.volcano_specific_genes/")
    mycolors <- c("blue", "red", "black")
    names(mycolors) <- c("DOWN", "UP", "NO")
    Log10Pvalue = -log10(data[[method]])
    data$specific_gene <- NA
    if (!is.null(specific_genes)) {
        data$specific_gene[data$gene_id %in% specific_genes] <- data$gene_id[data$gene_id %in% specific_genes]
    }
    Saving_file_name = paste0(Comparison_name)
    pdf(file= paste0(path,"1.volcano_specific_genes/",Saving_file_name,".pdf"), width=7, height=5)

    plot = ggplot(data=data, aes(x=log2FoldChange, y=Log10Pvalue, col=diffexpressed, label=specific_gene)) +
                geom_point() + 
                theme_minimal() +
                geom_text_repel() +
                scale_color_manual(values=mycolors) +
                geom_vline(xintercept=c(-FC, FC), col="red") +
                geom_hline(yintercept=-log10(pval), col="red") + 
                annotate("rect", xmin=Inf, xmax=Inf, ymin=Inf, ymax=Inf, fill="white", alpha=0.5) +
                annotate("text", x=Inf, y=Inf, label=paste("UP:", sum(data$diffexpressed == "UP")), vjust=3, hjust=1, size=3) +
                annotate("text", x=Inf, y=Inf, label=paste("DOWN:", sum(data$diffexpressed == "DOWN")), vjust=5, hjust=1, size=3) +
                ggtitle(paste0(Comparison_name)) + 
                theme(plot.title = element_text(hjust = 0.5)) + 
                ylab("-Log10 P-value") + 
                xlab("Log2 Fold Change")

    print(plot)
    dev.off()
}

# Pipeline

In [ ]:
make_folder = function(path,name){
    folder_name = paste0(path,name)
    if (!file.exists(folder_name)) {
        dir.create(folder_name)
    }
}
TMM_to_DESeq2_format = function(TMM_DGE_df){
    # 컬럼 이름 변경 및 순서 정렬 (DESeq2 스타일)
    colnames(TMM_DGE_df)[which(colnames(TMM_DGE_df) == "logFC")] <- "log2FoldChange"
    colnames(TMM_DGE_df)[which(colnames(TMM_DGE_df) == "PValue")] <- "pvalue"
    colnames(TMM_DGE_df)[which(colnames(TMM_DGE_df) == "FDR")] <- "padj"
    colnames(TMM_DGE_df)[which(colnames(TMM_DGE_df) == "gene_id")] <- "GENE_ID"    
    colnames(TMM_DGE_df)[which(colnames(TMM_DGE_df) == "gene_name")] <- "gene_id"
    TMM_DGE_df <- TMM_DGE_df[, c("GENE_ID", "gene_id", "logCPM", "log2FoldChange", "F", "pvalue", "padj")] # 여기 basemean 이랑 F 안바꿨다.
    return(TMM_DGE_df)
}
annotate_data = function(path, data, method, FC, pval, Comparison_name){
    result_data = data %>% filter(!is.na(padj))


    dge_description_df <- data.frame(
    Column = c("logCPM", "log2FoldChange", "F", "pvalue", "padj", "gene_id", "gene_name"),
    Meaning = c(
        "Log Normalized average expression value across all samples in two groups",
        "Log2-transformed fold change between two groups",
        "F-statistic from quasi-likelihood test from EdgeR",
        "Raw enrichment p-value with quasi-likelihood F-tests",
        "Adjusted p-value (false discovery rate (FDR))",
        "Ensembl gene id",
        "Gene Symbol"
    )
    )

    # write DGE data
    make_folder(path,"0.DGE/")

    # 여기 gene_id 되어 있는거 gene_name으로 바꿔서 excel 파일 저장. --> DESeq2랑은 호환 안되는 거임 여기까지는 아직.
    colnames(result_data)[which(colnames(result_data) == "gene_id")] <- "gene_name"
    out_list <- list(
        "Result" = result_data,
        "Column_Description" = dge_description_df
    )
    write_xlsx(out_list, paste0(path,"0.DGE/DEG_edgeR_",Comparison_name,".xlsx"))

    colnames(result_data)[which(colnames(result_data) == "gene_name")] <- "gene_id"
    result_data = result_data[, c("gene_id", "logCPM", "log2FoldChange", "F", "pvalue", "padj")]
    result_data$diffexpressed = "NO"
    result_data$diffexpressed[(result_data$log2FoldChange >= FC) & (result_data[[method]] <= pval)] <- "UP"
    result_data$diffexpressed[(result_data$log2FoldChange <= -FC) & (result_data[[method]] <= pval)] <- "DOWN"
    result_data$delabel <- NA
    result_data$delabel[result_data$diffexpressed != "NO"] <- result_data$gene_id[result_data$diffexpressed != "NO"]
    return(result_data)
}

analysis_data = function(path, Data, method, FC, pval, Comparison_name, Sample, Keytype, norm_method="edgeR", specific_geneset = NULL, specific_genes = NULL){ # SYMBOL # ENTREZID # ENSEMBL
    
    
    if (norm_method == "edgeR"){
        dge_data = TMM_to_DESeq2_format(Data)
    }
    else if( norm_method == "DESeq2"){
        dge_data = Data
    }

    #1)data
    data = annotate_data(path = path,data = dge_data, method = method,FC = FC,pval = pval,Comparison_name = Comparison_name)

    # volcano plot
    draw_volcanoplot(path,data,method,FC,pval,Comparison_name)

    # volcano plot with annotation
    draw_volcanoplot_annot(path,data,method,FC,pval,Comparison_name)

    # volcano plot with specific gene annotation
    draw_volcanoplot_genes(path,data,method,FC,pval,Comparison_name, specific_genes)

    #3)GO
    GO_DEG_analysis(path = path, data = data, Organism = Sample, Comparison_name = Comparison_name, Keytype = Keytype, Custom_geneset_gmt = specific_geneset)

    #4)GESA
    GSEA_analysis(path = path, data = data, Organism = Sample, Comparison_name = Comparison_name,Keytype = Keytype)

    #5)Specific GSEA
    run_specific_gsea_analysis(
        path = path,
        data = data,
        Comparison_name = Comparison_name,
        specific_geneset = specific_geneset,
        pvalue_cutoff = 1
    )
}

run_deg_pipeline <- function(project_settings, root_path) {
  prepare_inputs <- function(project_settings, root_path) {
    Project_name <- project_settings$Project_name
    comb_set <- project_settings$comb_set
    tx2gene_path <- project_settings$tx2gene_path
    metadata_path <- project_settings$metadata_path
    group_col <- project_settings$group_col

    if (is.null(group_col) || identical(group_col, "")) {
      stop("project_settings$group_col 을 설정해주세요.")
    }

    message("▶ Analyzing: ", Project_name)
    message("▶ Group column: ", group_col)
    make_folder(root_path, Project_name)
    result_path <- paste0(root_path, Project_name)

    tx2gene_data <- as.data.frame(
      read.csv(tx2gene_path, sep = "\t", col.names = c("tx_id", "gene_id", "gene_name"))
    )
    col_data <- as.data.frame(read.csv(metadata_path))

    if (!(group_col %in% colnames(col_data))) {
      stop("group_col이 metadata에 없습니다: ", group_col)
    }

    files <- setNames(col_data$file, col_data$sample)
    stopifnot(all(file.exists(files)))

    txi_salmon <- tximport(files, type = "salmon", tx2gene = tx2gene_data)
    col_data <- col_data[match(colnames(txi_salmon$counts), col_data$sample), ]

    tx2gene_unique <- tx2gene_data %>%
      dplyr::select(gene_id, gene_name) %>%
      dplyr::distinct(gene_id, .keep_all = TRUE)

    list(
      result_path = result_path,
      comb_set = comb_set,
      txi_salmon = txi_salmon,
      col_data = col_data,
      tx2gene_unique = tx2gene_unique,
      group_col = group_col
    )
  }

  save_tpm_tables <- function(txi_salmon, tx2gene_unique, result_path) {
    tpm_gene <- txi_salmon$abundance %>%
      as.data.frame() %>%
      rownames_to_column(var = "gene_id") %>%
      left_join(tx2gene_unique, by = "gene_id") %>%
      relocate(gene_id, .before = 1) %>%
      relocate(gene_name, .after = gene_id)

    write.csv(
      tpm_gene,
      file = paste0(result_path, "TPM_gene.csv"),
      row.names = FALSE
    )

    tpm_gene_log <- tpm_gene %>%
      mutate(across(where(is.numeric), ~ log2(.x + 1)))

    write.csv(
      tpm_gene_log,
      file = paste0(result_path, "TPM_gene_log2p1.csv"),
      row.names = FALSE
    )

    invisible(NULL)
  }

  build_edger_object <- function(txi_salmon, col_data, group_col) {
    cts <- txi_salmon$counts
    normMat <- txi_salmon$length

    normMat <- normMat / exp(rowMeans(log(normMat)))
    normCts <- cts / normMat
    eff.lib <- calcNormFactors(normCts) * colSums(normCts)

    normMat <- sweep(normMat, 2, eff.lib, "*")
    normMat <- log(normMat)

    y <- DGEList(cts)
    y <- scaleOffset(y, normMat)

    group_factor <- as.factor(col_data[[group_col]])
    design <- model.matrix(~ 0 + group_factor)
    colnames(design) <- levels(group_factor)

    list(y = y, design = design)
  }

  save_tmm_tables <- function(y, tx2gene_unique, result_path) {
    tmm <- edgeR::cpm(y, offset = y$offset, log = FALSE) %>%
      as.data.frame() %>%
      rownames_to_column(var = "gene_id") %>%
      left_join(tx2gene_unique, by = "gene_id") %>%
      relocate(gene_id, .before = 1) %>%
      relocate(gene_name, .after = gene_id)

    write.csv(
      tmm,
      file = paste0(result_path, "TMM_length_normalized_count.csv"),
      row.names = FALSE
    )

    log_tmm <- edgeR::cpm(y, offset = y$offset, log = TRUE) %>%
      as.data.frame() %>%
      rownames_to_column(var = "gene_id") %>%
      left_join(tx2gene_unique, by = "gene_id") %>%
      relocate(gene_id, .before = 1) %>%
      relocate(gene_name, .after = gene_id)

    write.csv(
      log_tmm,
      file = paste0(result_path, "log_TMM_length_normalized_count.csv"),
      row.names = FALSE
    )

    list(tmm = tmm, log_tmm = log_tmm)
  }

  run_deg_contrasts <- function(y, design, comb_set, tx2gene_unique) {
    dge_all <- estimateDisp(y, design)
    fit_all <- glmQLFit(dge_all, design)

    deg_result <- list()
    for (pair in comb_set) {
      treat <- pair[1]
      con <- pair[2]
      key_name <- paste0(treat, "_vs_", con)

      contrast_df <- makeContrasts(
        contrasts = paste0(treat, "-", con),
        levels = design
      )

      qlf <- glmQLFTest(fit_all, contrast = contrast_df)
      result_df <- as.data.frame(topTags(qlf, n = Inf))
      print(head(result_df))
      result_df$gene_id <- rownames(result_df)
      result_df <- result_df[, c("gene_id", setdiff(colnames(result_df), "gene_id"))]
      result_df <- result_df %>% left_join(tx2gene_unique, by = "gene_id")
      print(head(result_df))

      deg_result[[key_name]] <- result_df
    }
    return(deg_result)
  }

  prep <- prepare_inputs(project_settings, root_path)
  save_tpm_tables(prep$txi_salmon, prep$tx2gene_unique, prep$result_path)

  edge <- build_edger_object(prep$txi_salmon, prep$col_data, prep$group_col)
  tmm_tables <- save_tmm_tables(edge$y, prep$tx2gene_unique, prep$result_path)

  keep <- filterByExpr(edge$y, edge$design)
  y <- edge$y[keep, ]

  tmm_filtered <- edgeR::cpm(y, offset = y$offset, log = FALSE) %>%
    as.data.frame() %>%
    rownames_to_column(var = "gene_id") %>%
    left_join(prep$tx2gene_unique, by = "gene_id") %>%
    relocate(gene_id, .before = 1) %>%
    relocate(gene_name, .after = gene_id)

  write.csv(
    tmm_filtered,
    file = paste0(prep$result_path, "TMM_length_normalized_count_filtered.csv"),
    row.names = FALSE
  )

  TMM_length_normalized_count_log <- edgeR::cpm(y, offset = y$offset, log = TRUE) %>%
    as.data.frame() %>%
    rownames_to_column(var = "gene_id") %>%
    left_join(prep$tx2gene_unique, by = "gene_id") %>%
    relocate(gene_id, .before = 1) %>%
    relocate(gene_name, .after = gene_id)

  plot_pca_with_condition_and_batch(
    expr_matrix = TMM_length_normalized_count_log,
    col_data = prep$col_data,
    result_path = prep$result_path,
    condition_col_2 = prep$group_col,
    gene_annotation = prep$tx2gene_unique
  )

  dge_result <- run_deg_contrasts(y, edge$design, prep$comb_set, prep$tx2gene_unique)
  print(dge_result)
  # run_downstream_analysis(dge_result, prep$result_path)

  list(
    Project_name = project_settings$Project_name,
    group_col = prep$group_col,
    comb_set = prep$comb_set,
    result_path = prep$result_path,
    Col_data = prep$col_data,
    tx2gene_unique = prep$tx2gene_unique,
    y = y,
    DGE_result = dge_result,
    TMM_length_normalized_count = tmm_tables$tmm,
    log_TMM_length_normalized_count = tmm_tables$log_tmm,
    TMM_length_normalized_count_log = TMM_length_normalized_count_log
  )
}

# Settings

In [11]:
root_path = "./"

# 전체 설정 리스트화
project_settings <- list(
    Project_name = "test_run3/",
    group_col = "Condition",  # 비교군 기준 컬럼 (예: Condition, Group, Treatment)
    comb_set = list(
                    c("ConditionA", "ConditionB")
    ),
    tx2gene_path = "../1_Preprocessing/0_result/star_salmon/tx2gene.tsv",
    metadata_path = "./example_salmon_metadata.csv"
  )

# Run code

In [ ]:
pipeline_result <- run_deg_pipeline(
  project_settings = project_settings,
  root_path = root_path
)

group_col <- pipeline_result$group_col
result_path <- pipeline_result$result_path
Col_data <- pipeline_result$Col_data
tx2gene_unique <- pipeline_result$tx2gene_unique
y <- pipeline_result$y
DGE_result <- pipeline_result$DGE_result
TMM_length_normalized_count <- pipeline_result$TMM_length_normalized_count
log_TMM_length_normalized_count <- pipeline_result$log_TMM_length_normalized_count
TMM_length_normalized_count_log <- pipeline_result$TMM_length_normalized_count_log




▶ Analyzing: test_run3/

▶ Group column: Condition

reading in files with read_tsv

1 
2 
3 
4 
5 
6 


transcripts missing from tx2gene: 1

summarizing abundance

summarizing counts

summarizing length



                    logFC   logCPM        F       PValue          FDR
ENSG00000070808 14.324779 9.078875 6942.867 1.343265e-16 3.791411e-12
ENSG00000135439  7.617078 9.233880 5752.423 3.726294e-16 3.791411e-12
ENSG00000110436  8.768420 7.066367 5674.818 4.011165e-16 3.791411e-12
ENSG00000100884 13.695051 7.231197 5285.447 5.897734e-16 3.791411e-12
ENSG00000120251 13.592387 8.111440 5195.494 6.473077e-16 3.791411e-12
ENSG00000150625  6.823868 7.613787 4200.977 2.048255e-15 9.539699e-12
          gene_id     logFC   logCPM        F       PValue          FDR
1 ENSG00000070808 14.324779 9.078875 6942.867 1.343265e-16 3.791411e-12
2 ENSG00000135439  7.617078 9.233880 5752.423 3.726294e-16 3.791411e-12
3 ENSG00000110436  8.768420 7.066367 5674.818 4.011165e-16 3.791411e-12
4 ENSG00000100884 13.695051 7.231197 5285.447 5.897734e-16 3.791411e-12
5 ENSG00000120251 13.592387 8.111440 5195.494 6.473077e-16 3.791411e-12
6 ENSG00000150625  6.823868 7.613787 4200.977 2.048255e-15 9.539699e-12
  gene

In [13]:
for (comparison_name in names(DGE_result)) {
  analysis_data(
    path = result_path,
    Data = DGE_result[[comparison_name]],
    method = "padj",
    FC = 1,
    pval = 0.05,
    Comparison_name = comparison_name,
    Sample = "human",
    specific_geneset = "/octopus/yeongjun.kim/Reference/Specific/250609_MAPK_ERK_RAF_pathway_target.Hs.gmt",
    specific_genes = c("KRAS","ERBB2"),
    Keytype = "SYMBOL",
    norm_method = "edgeR"
  )
}

Warning message:
“Removed 10369 rows containing missing values or values outside the scale range
(`geom_text_repel()`).”
Warning message:
“ggrepel: 18910 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“Removed 29284 rows containing missing values or values outside the scale range
(`geom_text_repel()`).”
'select()' returned 1:many mapping between keys and columns

Warning message in bitr(symbols, fromType = keytype, toType = "ENTREZID", OrgDb = orgdb):
“26.96% of input gene IDs are fail to map...”
Reading KEGG annotation online: "https://rest.kegg.jp/link/hsa/pathway"...

Reading KEGG annotation online: "https://rest.kegg.jp/list/pathway/hsa"...

Reading KEGG annotation online: "https://rest.kegg.jp/conv/ncbi-geneid/hsa"...

'select()' returned 1:many mapping between keys and columns

Warning message in bitr(symbols, fromType = keytype, toType = "ENTREZID", OrgDb = orgdb):
“26.96% of input gene IDs are fail to map...”
'select()' returned 1: